In [1]:
# %pip install polars
# %pip install duckdb

In [2]:
import pandas as pd
import duckdb
import polars as pl

In [3]:
# Original code loading csv into parquet

# duckdb.sql("""
# COPY (
#     SELECT * FROM read_csv_auto('complaints.csv')
# )
# TO 'complaints.parquet'
# (FORMAT PARQUET);
# """)

In [4]:
# Load parquet into lazy dataframe and display the head
df = pl.scan_parquet("complaints.parquet")
df.head(5).collect()

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64
2020-07-06,"""Credit reporting, credit repai…","""Credit reporting""","""Incorrect information on your …","""Information belongs to someone…",null,"""Company has responded to the c…","""Experian Information Solutions…","""FL""","""346XX""",null,"""Other""","""Web""",2020-07-06,"""Closed with explanation""",true,"""N/A""",3730948
2019-12-26,"""Credit card or prepaid card""","""General-purpose credit card or…","""Advertising and marketing, inc…","""Confusing or misleading advert…",null,null,"""CAPITAL ONE FINANCIAL CORPORAT…","""CA""","""94025""",null,"""Consent not provided""","""Web""",2019-12-26,"""Closed with explanation""",true,"""N/A""",3477549
2020-05-08,"""Credit reporting, credit repai…","""Credit reporting""","""Incorrect information on your …","""Information belongs to someone…","""These are not my accounts.""","""Company has responded to the c…","""Experian Information Solutions…","""NV""","""89030""",null,"""Consent provided""","""Web""",2020-05-08,"""Closed with explanation""",true,"""N/A""",3642453
2024-01-05,"""Credit reporting or other pers…","""Credit reporting""","""Incorrect information on your …","""Information belongs to someone…","""Kindly address this issue on m…","""Company has responded to the c…","""Experian Information Solutions…","""IL""","""60502""",null,"""Consent provided""","""Web""",2024-01-05,"""Closed with non-monetary relie…",true,"""N/A""",8113747
2024-01-21,"""Credit reporting or other pers…","""Credit reporting""","""Improper use of your report""","""Credit inquiries on your repor…",null,"""Company has responded to the c…","""Experian Information Solutions…","""NC""","""27401""","""Servicemember""","""Consent not provided""","""Web""",2024-01-21,"""Closed with explanation""",true,"""N/A""",8191825


In [33]:
# Complaints by product
products = duckdb.sql("""
    SELECT
        Product,
        COUNT(*) AS complaints
    FROM 'complaints.parquet'
    GROUP BY Product
    ORDER BY complaints DESC
""")

products

┌──────────────────────────────────────────────────────────────────────────────┬────────────┐
│                                   Product                                    │ complaints │
│                                   varchar                                    │   int64    │
├──────────────────────────────────────────────────────────────────────────────┼────────────┤
│ Credit reporting or other personal consumer reports                          │    9570678 │
│ Credit reporting, credit repair services, or other personal consumer reports │    2163800 │
│ Debt collection                                                              │    1077479 │
│ Mortgage                                                                     │     446996 │
│ Checking or savings account                                                  │     361214 │
│ Credit card                                                                  │     307090 │
│ Credit card or prepaid card                               

In [11]:
# Number of records with non-null narratives and Yes/No for dispute
count_narratives_all = (
    df
    .filter(
        pl.col('Consumer complaint narrative').is_not_null()
        & pl.col('Consumer disputed?').is_in(['Yes', 'No'])
    )
    .select(pl.len().alias('count_narratives_all'))
    .collect()
)

print(f'The number of records with valid narratives AND yes/no in disputed: {count_narratives_all}')

The number of records with valid narratives AND yes/no in disputed: shape: (1, 1)
┌──────────────────────┐
│ count_narratives_all │
│ ---                  │
│ u32                  │
╞══════════════════════╡
│ 163960               │
└──────────────────────┘


In [51]:
# Number of records with non-null narratives - all products
count_narratives_all = (
    df
    .filter(
        pl.col('Consumer complaint narrative').is_not_null()
    )
    .select(pl.len().alias('count_narratives_all'))
    .collect()
)

print(f'The number of records with valid narratives: {count_narratives_all}')

The number of records with valid narratives: shape: (1, 1)
┌──────────────────────┐
│ count_narratives_all │
│ ---                  │
│ u32                  │
╞══════════════════════╡
│ 3770248              │
└──────────────────────┘


In [27]:
# Top 10 companies - valid narratives

duckdb.sql("""
    SELECT
        Company,
        COUNT(*) AS narrative_count
    FROM 'complaints.parquet'
    WHERE "Consumer complaint narrative" IS NOT NULL
    GROUP BY Company
    ORDER BY narrative_count DESC
    LIMIT 10
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────────────┐
│                Company                 │ narrative_count │
│                varchar                 │      int64      │
├────────────────────────────────────────┼─────────────────┤
│ EQUIFAX, INC.                          │          815798 │
│ TRANSUNION INTERMEDIATE HOLDINGS, INC. │          773419 │
│ Experian Information Solutions Inc.    │          736063 │
│ CAPITAL ONE FINANCIAL CORPORATION      │           68103 │
│ JPMORGAN CHASE & CO.                   │           62507 │
│ WELLS FARGO & COMPANY                  │           58537 │
│ BANK OF AMERICA, NATIONAL ASSOCIATION  │           56410 │
│ CITIBANK, N.A.                         │           53439 │
│ Block, Inc.                            │           46319 │
│ SYNCHRONY FINANCIAL                    │           33727 │
└────────────────────────────────────────┴─────────────────┘
  10 rows                                        2 columns

In [29]:
# Top 5 companies non-credit - valid narratives and Yes/No in dispute
duckdb.sql("""
    SELECT
        Company,
        COUNT(*) AS narrative_count
    FROM 'complaints.parquet'
    WHERE "Consumer complaint narrative" IS NOT NULL
      AND "Consumer disputed?" IN ('Yes', 'No')
      AND Company IN (
          'CAPITAL ONE FINANCIAL CORPORATION',
          'JPMORGAN CHASE & CO.',
          'WELLS FARGO & COMPANY',
          'BANK OF AMERICA, NATIONAL ASSOCIATION',
          'CITIBANK, N.A.'
      )
    GROUP BY Company
    ORDER BY narrative_count DESC
""")

┌───────────────────────────────────────┬─────────────────┐
│                Company                │ narrative_count │
│                varchar                │      int64      │
├───────────────────────────────────────┼─────────────────┤
│ WELLS FARGO & COMPANY                 │            7503 │
│ BANK OF AMERICA, NATIONAL ASSOCIATION │            7254 │
│ CITIBANK, N.A.                        │            6993 │
│ JPMORGAN CHASE & CO.                  │            6260 │
│ CAPITAL ONE FINANCIAL CORPORATION     │            3763 │
└───────────────────────────────────────┴─────────────────┘

In [32]:
# Filter the LazyFrame to Mortgage, Checking/Savings, Credit card, Bank account
bank_products_df = df.filter(
    pl.col('Product').is_in([
        'Mortgage',
        'Checking or savings account',
        'Credit card',
        'Credit card or prepaid card',
        'Bank account or service' 
    ])
)

bank_products_df.limit(5).collect()

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64
2019-12-26,"""Credit card or prepaid card""","""General-purpose credit card or…","""Advertising and marketing, inc…","""Confusing or misleading advert…",null,null,"""CAPITAL ONE FINANCIAL CORPORAT…","""CA""","""94025""",null,"""Consent not provided""","""Web""",2019-12-26,"""Closed with explanation""",true,"""N/A""",3477549
2019-12-20,"""Checking or savings account""","""Other banking product or servi…","""Managing an account""","""Funds not handled or disbursed…",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""FL""","""33064""",null,"""N/A""","""Referral""",2019-12-23,"""Closed with explanation""",true,"""N/A""",3475858
2019-09-04,"""Checking or savings account""","""Checking account""","""Managing an account""","""Banking errors""",null,null,"""MoneyLion Inc.""","""CA""","""94555""",null,"""Consent not provided""","""Web""",2019-09-04,"""Closed with monetary relief""",true,"""N/A""",3363680
2019-11-18,"""Credit card or prepaid card""","""General-purpose credit card or…","""Problem with a purchase shown …","""Credit card company isn't reso…","""XXXX claimed they delivered a …",null,"""DISCOVER BANK""","""MA""","""021XX""",null,"""Consent provided""","""Web""",2019-11-18,"""Closed with explanation""",true,"""N/A""",3442136
2020-06-05,"""Checking or savings account""","""Checking account""","""Managing an account""","""Problem using a debit or ATM c…",null,"""Company has responded to the c…","""CITIBANK, N.A.""","""NY""","""10466""",null,"""Consent not provided""","""Web""",2020-06-05,"""Closed with explanation""",true,"""N/A""",3684669


In [52]:
# Bank products with valid narratives and Yes/No dispute
count_narratives = (
    bank_products_df
    .filter(
        pl.col('Consumer complaint narrative').is_not_null()
    )
    .select(pl.len().alias('count_narratives'))
    .collect()
)

print(f'The number of records in bank_products with valid narratives: {count_narratives}')

The number of records in bank_products with valid narratives: shape: (1, 1)
┌──────────────────┐
│ count_narratives │
│ ---              │
│ u32              │
╞══════════════════╡
│ 556498           │
└──────────────────┘


In [31]:
# Filter the original LazyFrame to Mortage only
mortgage_df = df.filter(pl.col('Product') == 'Mortgage')

mortgage_df.head().collect()

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64
2019-01-09,"""Mortgage""","""Conventional home mortgage""","""Struggling to pay mortgage""",null,null,null,"""PNC Bank N.A.""",null,null,"""Servicemember""","""N/A""","""Phone""",2019-01-09,"""Closed with explanation""",true,"""N/A""",3120489
2014-03-27,"""Mortgage""","""Other mortgage""","""Loan servicing, payments, escr…",null,null,null,"""WELLS FARGO & COMPANY""","""CA""","""94611""",null,"""N/A""","""Referral""",2014-04-02,"""Closed with explanation""",true,"""No""",782303
2024-01-18,"""Mortgage""","""Conventional home mortgage""","""Trouble during payment process""","""Escrow, taxes, or insurance""",null,"""Company has responded to the c…","""NEW YORK COMMUNITY BANCORP INC""","""CT""","""06043""",null,"""Consent not provided""","""Web""",2024-01-18,"""Closed with explanation""",true,"""N/A""",8179245
2020-01-24,"""Mortgage""","""Other type of mortgage""","""Trouble during payment process""",null,null,"""Company has responded to the c…","""BANK OF AMERICA, NATIONAL ASSO…","""GA""","""30032""","""Older American""","""N/A""","""Referral""",2020-01-25,"""Closed with explanation""",true,"""N/A""",3509026
2019-08-15,"""Mortgage""","""Conventional home mortgage""","""Trouble during payment process""",null,null,"""Company has responded to the c…","""LoanCare, LLC""","""TX""","""75033""",null,"""Consent not provided""","""Web""",2019-08-20,"""Closed with explanation""",true,"""N/A""",3341452


In [53]:
# Print sample of 10 narratives
sample_narratives = (
    mortgage_df
    .filter(pl.col('Consumer complaint narrative').is_not_null())
    .select('Consumer complaint narrative')
    .collect()
    .sample(n=10, shuffle=True)
)

narratives = sample_narratives['Consumer complaint narrative'].to_list()

for i, narrative in enumerate(narratives, start=1):
    print(f'\n--- Narrative {i} ---')
    print(narrative)


--- Narrative 1 ---
I received a letter from an unknown bank, M & T bank, saying they are the alleged new servicer. I have not received any communication from the previous alleged servicer XXXX XXXX. Now I am getting numerous phone calls and mailings using my private intellectual property regarding the alleged mortgage, without my consent. list of calls and XXXX XXXX, XX/XX/year> XXXX, XX/XX/year> XXXX, XX/XX/year> XXXX, XX/XX/year> XXXX, XX/XX/year> XXXX.

--- Narrative 2 ---
XXXX : XXXX this agent persisted me on a daily basis and when I decided not to continue they pressurized me to complete assuring me they where best and would benefit me. I explained my concerns with closing costs and they assured me it was normal. I subsequently have been advised its above average. I was also told they would consolidate all our debt only to find most of it will be consumed by closing costs. Then insisted we close to lock in rate and now they going ahead and ensuring process would be efficient. S

In [16]:
# Save mortgage-only to parquet
# mortgage_df.sink_parquet('mortgage_only_complaints.parquet')

In [15]:
# Load parquet into pandas dataframe
pandas_mortgage_df = pd.read_parquet('mortgage_only_complaints.parquet')

In [17]:
pandas_mortgage_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 446996 entries, 0 to 446995
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 446996 non-null  object
 1   Product                       446996 non-null  object
 2   Sub-product                   446996 non-null  object
 3   Issue                         446996 non-null  object
 4   Sub-issue                     69135 non-null   object
 5   Consumer complaint narrative  141969 non-null  object
 6   Company public response       154269 non-null  object
 7   Company                       446996 non-null  object
 8   State                         441354 non-null  object
 9   ZIP code                      441915 non-null  object
 10  Tags                          86201 non-null   object
 11  Consumer consent provided?    441503 non-null  object
 12  Submitted via                 446996 non-null  object
 13 

In [18]:
# Percentage of records with non-blank/non-null narratives
count_non_null = pandas_mortgage_df['Consumer complaint narrative'].count()
print(f'The number of records in the mortgage df with non-blank/non-null narratives: {count_non_null}')
total_rows = len(pandas_mortgage_df)

pct_with_narratives = round(((count_non_null / total_rows) * 100), 2)
print(f'The percentage of records with non-blank complaint narratives is: {pct_with_narratives:.2f}%')

The number of records in the mortgage df with non-blank/non-null narratives: 141969
The percentage of records with non-blank complaint narratives is: 31.76%


In [19]:
# Value counts in "Consumer disputed?"
pandas_mortgage_df['Consumer disputed?'].value_counts()

Consumer disputed?
N/A    220156
No     175471
Yes     51369
Name: count, dtype: int64

In [46]:
# Value counts in "Timely"
pandas_mortgage_df['Timely response?'].value_counts()

Timely response?
True     438969
False      8027
Name: count, dtype: int64

In [20]:
# Percentage of records with valid yes/no in disputed column
non_na = pandas_mortgage_df[
    (pandas_mortgage_df['Consumer disputed?'] == 'Yes') | 
    (pandas_mortgage_df['Consumer disputed?'] == 'No')
    ]

count_non_na = len(non_na)
total_rows = len(pandas_mortgage_df)

pct_with_yes_no = round(((count_non_na / total_rows) * 100), 2)
print(f'The percentage of records with yes/no in disputed: {pct_with_yes_no:.2f}%')

# Count of records with yes/no disputed AND valid narrative
count_narratives = non_na['Consumer complaint narrative'].count()
print(f'The number of records with valid narratives AND yes/no in disputed: {count_narratives}')

The percentage of records with yes/no in disputed: 50.75%
The number of records with valid narratives AND yes/no in disputed: 32788


In [50]:
# Percentage of records with valid yes/no in disputed column
non_na = pandas_mortgage_df[
    (pandas_mortgage_df['Timely response?'] == True) | 
    (pandas_mortgage_df['Timely response?'] == False)
    ]

count_non_na = len(non_na)
total_rows = len(pandas_mortgage_df)

pct_with_yes_no = round(((count_non_na / total_rows) * 100), 2)
print(f'The percentage of records with yes/no in disputed: {pct_with_yes_no:.2f}%')

# Count of records with timely AND valid narrative
count_narratives = non_na['Consumer complaint narrative'].count()
print(f'The number of records with valid narratives AND yes/no in disputed: {count_narratives}')

The percentage of records with yes/no in disputed: 100.00%
The number of records with valid narratives AND yes/no in disputed: 141969


In [44]:
# Value counts in "Timely"
pandas_mortgage_df['Timely response?'].value_counts()

Timely response?
True     438969
False      8027
Name: count, dtype: int64

In [43]:
# Shift back to parquet file for SQL queries
# Count narratives by year - mortgage only
duckdb.sql("""
    SELECT
        YEAR("Date received") AS year, "Timely response?" as timely,
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM 'mortgage_only_complaints.parquet'
    GROUP BY year, timely
    ORDER BY year
""")

┌───────┬─────────┬────────────┬────────────┐
│ year  │ timely  │ complaints │ narratives │
│ int64 │ boolean │   int64    │   int64    │
├───────┼─────────┼────────────┼────────────┤
│  2011 │ false   │        171 │          0 │
│  2011 │ true    │       1105 │          0 │
│  2012 │ true    │      36336 │          0 │
│  2012 │ false   │       1772 │          0 │
│  2013 │ true    │      48902 │          0 │
│  2013 │ false   │        495 │          0 │
│  2014 │ false   │        846 │          0 │
│  2014 │ true    │      42095 │          0 │
│  2015 │ false   │        663 │        204 │
│  2015 │ true    │      41666 │      12097 │
│    ·  │  ·      │        ·   │        ·   │
│    ·  │  ·      │        ·   │        ·   │
│    ·  │  ·      │        ·   │        ·   │
│  2022 │ true    │      22893 │      12086 │
│  2022 │ false   │        393 │        199 │
│  2023 │ true    │      22616 │      12743 │
│  2023 │ false   │        238 │        140 │
│  2024 │ false   │        251 │  

In [40]:
# Shift back to parquet file for SQL queries
# Count narratives by year - all products
duckdb.sql("""
    SELECT
        YEAR("Date received") AS year, "Consumer disputed?" as dispute,
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM 'complaints.parquet'
    WHERE dispute = 'N/A'
    GROUP BY year, dispute
    ORDER BY year
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────┬────────────┬────────────┐
│ year  │ dispute │ complaints │ narratives │
│ int64 │ varchar │   int64    │   int64    │
├───────┼─────────┼────────────┼────────────┤
│  2017 │ N/A     │     170469 │      83614 │
│  2018 │ N/A     │     257137 │     118393 │
│  2019 │ N/A     │     277249 │     124836 │
│  2020 │ N/A     │     444231 │     174306 │
│  2021 │ N/A     │     495956 │     203564 │
│  2022 │ N/A     │     800312 │     337165 │
│  2023 │ N/A     │    1292069 │     487417 │
│  2024 │ N/A     │    2734293 │     814390 │
│  2025 │ N/A     │    5442769 │    1221783 │
│  2026 │ N/A     │    2207174 │      40820 │
└───────┴─────────┴────────────┴────────────┘
  10 rows                         4 columns

In [40]:
# Shift back to parquet file for SQL queries
# Count narratives by year - all products
duckdb.sql("""
    SELECT
        YEAR("Date received") AS year, "Consumer disputed?" as dispute,
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM 'complaints.parquet'
    WHERE dispute = 'N/A'
    GROUP BY year, dispute
    ORDER BY year
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────┬────────────┬────────────┐
│ year  │ dispute │ complaints │ narratives │
│ int64 │ varchar │   int64    │   int64    │
├───────┼─────────┼────────────┼────────────┤
│  2017 │ N/A     │     170469 │      83614 │
│  2018 │ N/A     │     257137 │     118393 │
│  2019 │ N/A     │     277249 │     124836 │
│  2020 │ N/A     │     444231 │     174306 │
│  2021 │ N/A     │     495956 │     203564 │
│  2022 │ N/A     │     800312 │     337165 │
│  2023 │ N/A     │    1292069 │     487417 │
│  2024 │ N/A     │    2734293 │     814390 │
│  2025 │ N/A     │    5442769 │    1221783 │
│  2026 │ N/A     │    2207174 │      40820 │
└───────┴─────────┴────────────┴────────────┘
  10 rows                         4 columns

In [40]:
# Shift back to parquet file for SQL queries
# Count narratives by year - all products
duckdb.sql("""
    SELECT
        YEAR("Date received") AS year, "Timely?" as timely,
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM 'complaints.parquet'
    WHERE dispute = 'N/A'
    GROUP BY year, dispute
    ORDER BY year
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────┬────────────┬────────────┐
│ year  │ dispute │ complaints │ narratives │
│ int64 │ varchar │   int64    │   int64    │
├───────┼─────────┼────────────┼────────────┤
│  2017 │ N/A     │     170469 │      83614 │
│  2018 │ N/A     │     257137 │     118393 │
│  2019 │ N/A     │     277249 │     124836 │
│  2020 │ N/A     │     444231 │     174306 │
│  2021 │ N/A     │     495956 │     203564 │
│  2022 │ N/A     │     800312 │     337165 │
│  2023 │ N/A     │    1292069 │     487417 │
│  2024 │ N/A     │    2734293 │     814390 │
│  2025 │ N/A     │    5442769 │    1221783 │
│  2026 │ N/A     │    2207174 │      40820 │
└───────┴─────────┴────────────┴────────────┘
  10 rows                         4 columns

In [21]:
# Shift back to parquet file for SQL queries
# Count narratives by year - mortgage only - timely
duckdb.sql("""
    SELECT
        YEAR("Date received") AS year,
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM 'mortgage_only_complaints.parquet'
    GROUP BY year
    ORDER BY year
""")

┌───────┬────────────┬────────────┐
│ year  │ complaints │ narratives │
│ int64 │   int64    │   int64    │
├───────┼────────────┼────────────┤
│  2011 │       1276 │          0 │
│  2012 │      38108 │          0 │
│  2013 │      49397 │          0 │
│  2014 │      42941 │          0 │
│  2015 │      42329 │      12301 │
│  2016 │      41455 │      15765 │
│  2017 │      30559 │      13137 │
│  2018 │      24568 │      10270 │
│  2019 │      22701 │      10033 │
│  2020 │      24652 │      12467 │
│  2021 │      26533 │      14080 │
│  2022 │      23286 │      12285 │
│  2023 │      22854 │      12883 │
│  2024 │      21449 │      12212 │
│  2025 │      24661 │      13992 │
│  2026 │      10227 │       2544 │
└───────┴────────────┴────────────┘
  16 rows               3 columns

In [22]:
# Narratives by sub-product - mortgage only
duckdb.sql("""
    SELECT
        "Sub-product",
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM 'mortgage_only_complaints.parquet'
    GROUP BY "Sub-product"
    ORDER BY narratives DESC
""")

┌────────────────────────────────────────────┬────────────┬────────────┐
│                Sub-product                 │ complaints │ narratives │
│                  varchar                   │   int64    │   int64    │
├────────────────────────────────────────────┼────────────┼────────────┤
│ Conventional home mortgage                 │     131596 │      64855 │
│ FHA mortgage                               │      60692 │      26569 │
│ Conventional fixed mortgage                │      70593 │      14557 │
│ VA mortgage                                │      22012 │      11597 │
│ Home equity loan or line of credit (HELOC) │      13749 │       6187 │
│ Other type of mortgage                     │      17521 │       5672 │
│ Conventional adjustable mortgage (ARM)     │      25380 │       4975 │
│ Other mortgage                             │      86612 │       3236 │
│ Home equity loan or line of credit         │      11622 │       2102 │
│ Reverse mortgage                           │     

In [23]:
# Min/Max and Avg characters in the narratives
duckdb.sql("""
SELECT
    MIN(LENGTH("Consumer complaint narrative")) AS min_chars,
    AVG(LENGTH("Consumer complaint narrative")) AS avg_chars,
    MAX(LENGTH("Consumer complaint narrative")) AS max_chars
FROM 'mortgage_only_complaints.parquet'
WHERE "Consumer complaint narrative" IS NOT NULL;
""")

┌───────────┬────────────────────┬───────────┐
│ min_chars │     avg_chars      │ max_chars │
│   int64   │       double       │   int64   │
├───────────┼────────────────────┼───────────┤
│        13 │ 1666.9142066225725 │     32317 │
└───────────┴────────────────────┴───────────┘

In [24]:
# Top 20 longest narratives with product and sub-product
duckdb.sql("""
SELECT
    LENGTH("Consumer complaint narrative") AS chars,
    "Issue",
    "Sub-product"
FROM 'mortgage_only_complaints.parquet'
WHERE "Consumer complaint narrative" IS NOT NULL
ORDER BY chars DESC
LIMIT 20;
""")

┌───────┬─────────────────────────────────────────────────────────────┬────────────────────────────┐
│ chars │                            Issue                            │        Sub-product         │
│ int64 │                           varchar                           │          varchar           │
├───────┼─────────────────────────────────────────────────────────────┼────────────────────────────┤
│ 32317 │ Closing on a mortgage                                       │ Other type of mortgage     │
│ 31922 │ Trouble during payment process                              │ FHA mortgage               │
│ 31737 │ Trouble during payment process                              │ Conventional home mortgage │
│ 31645 │ Trouble during payment process                              │ FHA mortgage               │
│ 31636 │ Trouble during payment process                              │ Conventional home mortgage │
│ 31611 │ Struggling to pay mortgage                                  │ FHA mortgage       

In [25]:
# Narratives by issue
duckdb.sql("""
SELECT
    Issue,
    COUNT(*) AS narratives
FROM 'mortgage_only_complaints.parquet'
WHERE "Consumer complaint narrative" IS NOT NULL
GROUP BY Issue
ORDER BY narratives DESC;
""")

┌──────────────────────────────────────────────────────────────────────────────────┬────────────┐
│                                      Issue                                       │ narratives │
│                                     varchar                                      │   int64    │
├──────────────────────────────────────────────────────────────────────────────────┼────────────┤
│ Trouble during payment process                                                   │      54142 │
│ Struggling to pay mortgage                                                       │      26004 │
│ Loan servicing, payments, escrow account                                         │      14720 │
│ Applying for a mortgage or refinancing an existing mortgage                      │      14534 │
│ Loan modification,collection,foreclosure                                         │      10788 │
│ Closing on a mortgage                                                            │      10481 │
│ Application, origi